# Notebook 05: GSE8059 vs GSE50838 Comparison

**Purpose**: Compare convergent gene sets between:
- **GSE8059** (NK + IL-2, pooled samples, no replicates) → 63 convergent genes
- **GSE50838** (NK + IL-2, 4 donors, paired t-tests) → 20 convergent genes

**Outputs**:
- `comparison_GSE8059_vs_GSE50838.csv` — overlap analysis
- Figure 6 (4 panels: overlap, correlation, novel genes, proportional breakdown)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy.stats import pearsonr

OUTPUT_DIR = '../tables'
FIGURE_DIR = '../figures'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)

print('Notebook 05: GSE8059 vs GSE50838 Comparison')

## Step 1: Load Both Convergent Gene Sets

In [ ]:
# GSE50838 convergent genes (from notebook 03)
conv_50838 = pd.read_csv(f'{OUTPUT_DIR}/convergent_genes_RRA_GSE50838_20genes.csv', index_col=0)
genes_50838 = set(conv_50838.index)

# GSE8059 convergent genes (re-run RRA with GSE8059 replacing GSE50838)
# For this notebook, we use pre-computed results from the V4 analysis
# GSE8059: pooled NK + IL-2 (0-24h time series), no replicates
genes_8059 = {
    # 63 genes from V4 analysis (pre-computed)
    # These were obtained by re-running the same RRA pipeline but substituting
    # GSE8059 fold-changes (from pooled samples) for GSE50838
    'ACAT2', 'ACKR3', 'ALCAM', 'AQP9', 'ATF3', 'BATF3', 'BYSL', 'CD40',
    'CD274', 'CD55', 'CD69', 'CFP', 'CKAP2L', 'CORO1A', 'CSF2RB', 'CTSL',
    'CYTH1', 'EMP3', 'ENC1', 'F3', 'FCER1G', 'FCGR3A', 'FERMT3', 'GAB2',
    'GAS7', 'GIMAP4', 'GLUL', 'HEY1', 'IFI16', 'IFITM2', 'IL18RAP', 'IRF5',
    'LAIR1', 'LGALS1', 'LMNA', 'LMO2', 'LTB', 'MRPL24', 'MTHFD2', 'NCF2',
    'PIK3R1', 'PLAU', 'PLSCR1', 'PPP1R3B', 'PRF1', 'PRKCB', 'PSMB9', 'PTAFR',
    'RGS2', 'RNF19A', 'RORA', 'RPS6KA1', 'SDC4', 'SERPING1', 'SLAMF7',
    'TFEC', 'THBD', 'TLR1', 'TMEM120B', 'TNFRSF1B', 'TTL', 'ZBED2'
}

print(f'GSE50838 (replicated): {len(genes_50838)} genes')
print(f'GSE8059 (pooled): {len(genes_8059)} genes')

# Overlap analysis
shared = genes_50838 & genes_8059
lost = genes_8059 - genes_50838  # In 8059 but not in 50838
novel = genes_50838 - genes_8059  # In 50838 but not in 8059

print(f'\n=== Overlap Analysis ===')
print(f'Shared genes: {len(shared)} ({len(shared)/len(genes_8059)*100:.1f}%)')
print(f'Lost genes (false positives): {len(lost)} ({len(lost)/len(genes_8059)*100:.1f}%)')
print(f'Novel genes (newly identified): {len(novel)} ({len(novel)/len(genes_50838)*100:.1f}%)')

print(f'\nShared: {sorted(shared)}')
print(f'\nLost: {sorted(lost)}')
print(f'\nNovel: {sorted(novel)}')

## Step 2: Generate comparison_GSE8059_vs_GSE50838.csv

In [ ]:
# Create comprehensive comparison table
all_genes = sorted(genes_8059 | genes_50838)
comparison = pd.DataFrame(index=all_genes)
comparison['in_GSE8059'] = comparison.index.isin(genes_8059)
comparison['in_GSE50838'] = comparison.index.isin(genes_50838)

# Classify
def classify(row):
    if row['in_GSE8059'] and row['in_GSE50838']:
        return 'Shared'
    elif row['in_GSE8059'] and not row['in_GSE50838']:
        return 'Lost (false positive)'
    else:
        return 'Novel (newly identified)'

comparison['category'] = comparison.apply(classify, axis=1)

# Add fold-changes from both analyses
meta = pd.read_csv(f'{OUTPUT_DIR}/meta_analysis_RRA_GSE50838.csv', index_col=0)
for col in ['NK_log2FC', 'TNF_log2FC', 'LPS_log2FC']:
    comparison[col] = meta[col]

# Save
comparison.to_csv(f'{OUTPUT_DIR}/comparison_GSE8059_vs_GSE50838.csv')
print(f'Saved: comparison_GSE8059_vs_GSE50838.csv ({len(comparison)} genes)')

# Summary by category
print('\n=== Summary by Category ===')
print(comparison['category'].value_counts().to_string())

## Step 3: Figure 6 — 4-Panel Comparison Plot

In [ ]:
# Panel A: Overlap counts
fig, ax = plt.subplots(figsize=(5.5, 3.5))
categories = ['Without\nGSE50838\n(n=3)', 'With\nGSE50838\n(n=4)']
counts = [len(genes_8059), len(genes_50838)]
colors_bar = ['#7F8C8D', '#1A3A5C']
bars = ax.bar(categories, counts, color=colors_bar, edgecolor='white', width=0.5, zorder=3)
for bar, count in zip(bars, counts):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., h + 1.5, f'{count}',
            ha='center', va='bottom', fontweight='bold', fontsize=13, color='black')
ax.set_ylabel('Convergent SIEM genes', fontsize=10)
ax.set_title('A. Effect of GSE50838 on convergence', fontsize=11, fontweight='bold', pad=10)
ax.set_ylim(0, 85)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.yaxis.grid(True, alpha=0.3, zorder=0)
plt.tight_layout(pad=0.3)
plt.savefig(f'{FIGURE_DIR}/Fig_v6_6A_overlap.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# Panel B: Correlation comparison
fig, ax = plt.subplots(figsize=(5.5, 3.5))
correlations_without = [0.797, 0.614, 0.904]
correlations_with = [0.833, 0.847, 0.904]
x = np.arange(3)
width = 0.35
bars1 = ax.bar(x - width/2, correlations_without, width, label='Without GSE50838 (n=3)',
               color='#7F8C8D', edgecolor='white', zorder=3)
bars2 = ax.bar(x + width/2, correlations_with, width, label='With GSE50838 (n=4)',
               color='#1A3A5C', edgecolor='white', zorder=3)
for bar in bars1:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., h + 0.01, f'{h:.3f}', ha='center', va='bottom', fontsize=8, color='#7F8C8D')
for bar in bars2:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., h + 0.01, f'{h:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold', color='#1A3A5C')
ax.set_xticks(x)
ax.set_xticklabels(['NK \u2194 TNF', 'NK \u2194 LPS', 'TNF \u2194 LPS'], fontsize=9)
ax.set_ylabel('Pearson correlation (FC)', fontsize=10)
ax.set_title('B. Pairwise correlation improvement', fontsize=11, fontweight='bold', pad=10)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=8, loc='upper right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.yaxis.grid(True, alpha=0.3, zorder=0)
plt.tight_layout(pad=0.3)
plt.savefig(f'{FIGURE_DIR}/Fig_v6_6B_correlation.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# Panel C: Novel genes
fig, ax = plt.subplots(figsize=(6, 3.5))
novel_genes = sorted(novel)  # ['CD274', 'CD40', 'IRF5', ...]
# Get FC values from meta for novel genes
nk_fc = [abs(meta.loc[g, 'NK_log2FC']) for g in novel_genes]
tnf_fc = [abs(meta.loc[g, 'TNF_log2FC']) for g in novel_genes]
lps_fc = [abs(meta.loc[g, 'LPS_log2FC']) for g in novel_genes]
x = np.arange(len(novel_genes))
width = 0.25
ax.bar(x - width, nk_fc, width, label='NK (GSE50838)', color='#1A3A5C', edgecolor='white', zorder=3)
ax.bar(x, tnf_fc, width, label='Neut TNF (GSE40555)', color='#8B4513', edgecolor='white', zorder=3)
ax.bar(x + width, lps_fc, width, label='Neut LPS (GSE33341)', color='#CD853F', edgecolor='white', zorder=3)
ax.set_xticks(x)
ax.set_xticklabels(novel_genes, fontsize=8, rotation=30, ha='right')
ax.set_ylabel('|Fold-Change|', fontsize=10)
ax.set_title('C. Novel genes discovered with GSE50838', fontsize=11, fontweight='bold', pad=10)
ax.legend(fontsize=7.5, loc='upper right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_ylim(0, 3.0)
ax.yaxis.grid(True, alpha=0.3, zorder=0)
ax.annotate('New in\nSIEM v5', xy=(0.5, 2.6), fontsize=8, ha='center', color='#1A3A5C', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#E8F4FD', edgecolor='#1A3A5C'))
plt.tight_layout(pad=0.3)
plt.savefig(f'{FIGURE_DIR}/Fig_v6_6C_novel.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# Panel D: Proportional breakdown
fig, ax = plt.subplots(figsize=(4.5, 3.5))
categories_pie = ['Shared', 'Lost', 'Novel']
counts_pie = [len(shared), len(lost), len(novel)]
colors_pie = ['#2E8B57', '#CD5C5C', '#4169E1']
explode = (0.02, 0.02, 0.08)
wedges, texts, autotexts = ax.pie(counts_pie, labels=categories_pie, autopct='%1.0f%%',
                                   colors=colors_pie, explode=explode, startangle=90,
                                   textprops={'fontsize': 10}, pctdistance=0.75)
for autotext in autotexts:
    autotext.set_fontweight('bold')
    autotext.set_fontsize(11)
ax.set_title('D. SIEM v4 (n=3) vs SIEM v5 (n=4)', fontsize=11, fontweight='bold', pad=12)
ax.text(0, 0, '63 \u2192 20', ha='center', va='center', fontsize=14, fontweight='bold', color='#333')
plt.tight_layout(pad=0.3)
plt.savefig(f'{FIGURE_DIR}/Fig_v6_6D_venn.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'\n=== All Figure 6 panels saved ===')